# BioJEPA v0.7 -- Closed-form LSQ Decoder

Fits the shared read-out head in closed form (ridge least-squares) over one full **balanced** epoch
instead of SGD, then runs the AC evals on the **test** split. Kept self-contained here until the head
is validated (nothing added to core `training_v0_7.py`). The full balanced epoch lets the loader's shard
balancing set dataset representation, so no manual stratification is needed.

`USE_TEACHER_DELTA` switches the fit target: `False` = predictor deltas (deploy-optimal readout of the
actual predictor output), `True` = teacher deltas (clean, predictor-agnostic benchmark probe).

In [ ]:
import torch
import gc
import random
from pathlib import Path

import numpy as np

from biojepa_v0_7 import BioJepa, BioJepaConfig
from dataloader_v0_7 import TrainingLoader
from training_v0_7 import create_model, maybe_compile, load_feature_banks, get_seq_embeddings, get_target_embeddings, reset_seed
from config_v0_7 import DataConfig, VERSION
from evals.evals import EvalContext, run_ac_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

## Device & Paths

In [ ]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()

data_root = Path('~/data/v0_7').expanduser()
ref_root = Path('~/data/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'short_checkpoint',
    ref_dir=ref_root,
    eval_results_dir=data_root / 'short_eval_results',
)

# Fit knobs
RIDGE = 1e-2
USE_TEACHER_DELTA = False   # False = predictor deltas (deploy); True = teacher deltas (benchmark)
DECODER_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 32

## Build Model & Load AC Checkpoint

Same config as the short-path training (decoupled 128/4/4 predictor), so the `biojepa_v0_7_ac_final.pt`
state dict loads strict.

In [ ]:
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=6,
    heads=4,
    embed_dim=256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=5.699,
    film_linear_multiple=0.6769,
    sim_coeff=50.18,
    std_coeff=25.44,
    cov_coeff=0.5158,
    pert_latent_dim=128,
    pert_mode_dim=64,
    predictor_embed_dim=128,
    predictor_n_layer=4,
    predictor_heads=4,
)

model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)
seq_banks, target_bank = load_feature_banks(data_cfg, device)

print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters()):,}')

In [ ]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_ac_final.pt'
with torch.serialization.safe_globals([BioJepaConfig]):
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

## Fit the Closed-form LSQ Decoder (one full balanced epoch)

Solves `W = (sum dz dz^T + ridge*I)^-1 (sum dz dy)` where `dz` is the latent delta and `dy` the real
expression delta, over measured genes only. Forward + matmul only (no backprop), deterministic.

In [ ]:
def fit_linear_decoder(model, train_loader, seq_banks, target_bank, model_cfg, device, ridge=1e-2, use_amp=False, use_teacher_delta=False):
    '''Closed-form ridge fit of the shared 256->1 read-out head over one full balanced epoch.
    Solves min_W sum_(measured g) (W . dz_g - dy_g)^2 + ridge * ||W||^2 via the normal equations.
    dz = predictor latent delta (z_pred_mu - z_context), or teacher delta when use_teacher_delta.'''
    reset_seed()
    use_autocast = use_amp and device.type == 'cuda'
    D = model_cfg.embed_dim
    steps_per_epoch = train_loader.total_samples // train_loader.batch_size
    delta_src = 'teacher' if use_teacher_delta else 'predictor'
    print(f'Decoder fit ({delta_src} delta): {train_loader.total_samples} samples, {steps_per_epoch} steps (one balanced epoch)')

    A = torch.zeros(D, D, dtype=torch.float64, device=device)
    Bvec = torch.zeros(D, dtype=torch.float64, device=device)

    model.eval()
    for param in model.parameters():
        param.requires_grad = False

    with torch.no_grad():
        for step in range(steps_per_epoch):
            b = train_loader.next_batch()
            B, N = b.control.shape
            seq_emb = get_seq_embeddings(b.seq_idx, b.modality, seq_banks)
            target_emb = get_target_embeddings(b.target_idx, target_bank)
            pert_mask = torch.arange(b.seq_idx.shape[1], device=device).unsqueeze(0) < b.n_perts.unsqueeze(1)
            unknown_mask = ~b.gene_mask

            with torch.autocast('cuda', dtype=torch.bfloat16, enabled=use_autocast):
                z_context = model.teacher(b.control, b.control_total, mask_idx=None, unknown_mask=unknown_mask)
                if use_teacher_delta:
                    z_delta_src = model.teacher(b.case, b.case_total, mask_idx=None, unknown_mask=unknown_mask)
                else:
                    action_latents = model.composer(seq_emb, target_emb, b.modality, b.mode, b.has_seq, b.has_target, pert_mask, dose=b.dose)
                    target_indices = torch.arange(N, device=device).expand(B, N)
                    z_delta_src, _ = model.predictor(z_context, action_latents, target_indices)

            dz = (z_delta_src.float() - z_context.float()).reshape(-1, D)
            dy = (b.case - b.control).reshape(-1).float()
            measured = b.gene_mask.reshape(-1)
            dz, dy = dz[measured].double(), dy[measured].double()
            A += dz.T @ dz
            Bvec += dz.T @ dy

            if step % 1000 == 0:
                print(f'  fit step {step}/{steps_per_epoch}')

    A += ridge * torch.eye(D, dtype=torch.float64, device=device)
    W = torch.linalg.solve(A, Bvec).float()

    decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=D)).to(device)
    decoder.head.weight.data = W.reshape(1, D)
    decoder.head.bias.data.zero_()
    decoder.eval()
    print(f'Fit W norm={float(W.norm()):.4f}')
    return decoder, W

In [ ]:
decoder_train_loader = TrainingLoader(
    batch_size=DECODER_BATCH_SIZE,
    split='train', data_dir=data_cfg.data_root / 'predictor_t',
    device=device)

decoder, W = fit_linear_decoder(model, decoder_train_loader, seq_banks, target_bank, model_cfg, device,
                                ridge=RIDGE, use_amp=USE_AMP, use_teacher_delta=USE_TEACHER_DELTA)

torch.save({'model': decoder.state_dict(), 'W': W.cpu(), 'ridge': RIDGE, 'use_teacher_delta': USE_TEACHER_DELTA},
           data_cfg.checkpoint_dir / f'biojepa_{VERSION}_decoder_lsq_final.pt')

del decoder_train_loader
gc.collect()
torch.cuda.empty_cache()

## AC Evals (test split)

`run_ac_evals` defaults to the **test** split. Report saved separately (`_lsq_decoder`) so it can be
diffed against the SGD decoder's `ac_eval_report.json`.

The decoder is applied *inside* `_run_test_inference` and the decoded deltas are cached, so the cache
**must** be cleared for the new head to take effect -- otherwise the eval reloads the old decoder's
cached deltas and the LSQ head has no effect.

In [ ]:
import shutil
cache_dir = data_cfg.data_root / 'test_inference_cache'
shutil.rmtree(cache_dir, ignore_errors=True)
print(f'cleared {cache_dir}')

In [ ]:
eval_config = {
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
}
model.eval()
eval_ctx = EvalContext(config=eval_config, data_root=data_cfg.data_root, checkpoint_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir)
eval_ctx._biojepa = model
eval_ctx._decoder = decoder
ac_eval_results = run_ac_evals(eval_ctx)
eval_ctx._biojepa = None

save_report(ac_eval_results, data_cfg.eval_results_dir / 'ac_eval_report_lsq_decoder.json')
ac_eval_results

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()